In [4]:
# Converte Latitude e Longitude de Graus para Decimais
def converter_latitude_longitude(graus_str):
    try:
        # Remove espaços extras e separa os componentes
        partes = graus_str.replace(' ', '').split('°')
        graus = int(partes[0])

        minutos_segundos = partes[1].split("'")
        minutos = int(minutos_segundos[0])

        segundos_str = minutos_segundos[1].replace("''", "")
        segundos = float(segundos_str)

        # Determina o sinal com base na direçãoo (S/W é negativo)
        direcao = minutos_segundos[1][-1] # Pega o último caractere (S ou W)
        if direcao in ('S', 'W'):
            return graus + (minutos / 60) + (segundos / 3600)
        elif direcao in ('N', 'E'):
            # Norte e Leste são positivos em muitas convenções, mas para simplificar a conversão para negativo, usamos aqui o sinal oposto. Ajuste se precisar de outra convenção.
            return -(graus + (minutos / 60) + (segundos / 3600))
        else:
            return None

    except (ValueError, IndexError):
        # print(f"Erro: Formato inválido para a entrada '{graus_str}'. Use 'DD° MM' SS'' [N/S/E/W]'.")
        return None

In [5]:
# Calcular a distancia entre aeroportos
def calcular_distancia_entre_aeroportos(lat1, lon1, lat2, lon2):
    # Raio médio da Terra em quilômetros
    R = 6371.0

    # Converte as latitudes e longitudes de graus para radianos
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Diferenças nas latitudes e longitudes
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    # Fórmula de Haversine
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Distância final
    distancia_km = R * c
    return distancia_km


In [6]:
# detectar o encoding do arquivo a ser lido
def detectar_encoding(caminho_arquivo):
    with open(caminho_arquivo, 'rb') as f:
        resultado = chardet.detect(f.read())
    return resultado['encoding']


In [ ]:
def ler_arquivo_bronze(diretorio, nome_arquivo):
    # 1. Monta o caminho completo
    caminho_completo = os.path.join(diretorio, nome_arquivo)

    # 2. Valida se o arquivo existe fisicamente antes de tentar ler
    if not os.path.exists(caminho_completo):
        return(f"❌ ERRO: O arquivo '{nome_arquivo}' não foi encontrado no diretório '{diretorio}'.")

    # 3. Bloco Try para tratamento de erros de leitura
    try:
        # Verifica a extensão para usar o comando correto
        if nome_arquivo.endswith('.csv'):
             df = pd.read_csv(caminho_completo, sep=';', encoding=detectar_encoding(caminho_completo))
        elif nome_arquivo.endswith(('.xls', '.xlsx')):
            df = pd.read_excel(caminho_completo)
        else:
            return("⚠️ Formato de arquivo não suportado.")
            return None
        return df

    except Exception as e:
        return(f"❌ Falha crítica ao processar o arquivo: {e}")